## Image Support

Claude can look at images as well as read text. You send an image inside a user message and ask a question about it. Claude can describe it, read text in it, compare images, and reason about what it shows.

In this notebook we send a satellite photo of a property and ask Claude to assess its wildfire risk.

### How an image is sent

A message's `content` is normally a string. To include an image, make `content` a **list of blocks** instead:

1. An `image` block that holds the picture. We send it as **base64**, a way of writing the file's bytes as text.
2. A `text` block with your question or instructions.

Each image block states its `media_type` (here `image/png`) so Claude knows how to read the data.

In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

**What this cell does:** Loads your API key, creates the client, and picks the model.

- `import base64` is used below to turn the file into text that can be sent in a request.
- `load_dotenv()` reads `ANTHROPIC_API_KEY` from your `.env` file, and `Anthropic()` finds it automatically.
- `model = "claude-haiku-5-5"` is the model used for every request in this notebook.

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    effort=None,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    # Haiku 5.5 thinks adaptively by default; effort sets how hard it thinks
    if effort:
        params["output_config"] = {"effort": effort}

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

**What this cell does:** Defines the helper functions used in the rest of the notebook.

- `add_user_message` and `add_assistant_message` add one message to the conversation. The `content` can be plain text or a list of content blocks, which is how you send a file together with a question.
- `chat` sends the conversation to Claude and returns the whole response. `effort` is optional and sets how hard Claude thinks. `max_tokens` is `4000` because thinking counts toward that limit.
- `text_from_message` returns only the text blocks of a response, skipping any thinking block.

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

**What this cell does:** Stores the instructions for the task in a variable called `prompt`.

- The prompt asks Claude to work through five steps: find the main house, check which trees overhang the roof, assess the fire risk, look at the space around the home, and give a rating from 1 to 4.
- Step-by-step instructions with a clear scoring scale make the answer more consistent and easier to check.
- The prompt ends by asking for one sentence per step and a final numeric rating, so the output has a predictable shape.
- Nothing is sent to Claude yet. This cell only creates the text.

In [7]:
# TODO: Read image data, feed into Claude
with open("images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text":prompt,
         
    }
])
chat_response = chat(messages)

**What this cell does:** Sends the image and the prompt to Claude.

- `open("images/prop7.png", "rb")` reads the picture as raw bytes. `base64.standard_b64encode(...).decode("utf-8")` turns the bytes into a base64 text string that can travel inside the request.
- The user message is a **list of two blocks**. The `image` block carries the data, with `"type": "base64"` and `"media_type": "image/png"`. The `text` block carries the `prompt`.
- The image comes before the text. Putting the picture first and the question after it is the usual order.
- `chat(messages)` sends the request, and the whole response is saved in `chat_response`.

> **Note:** Change the media type to match the file. Use `image/jpeg` for `.jpg` files, for example. A wrong media type can cause an error.

In [8]:
print(text_from_message(chat_response))

1. **Residence identification:** The primary residence is the largest gray-roofed structure, a two-story cross-gabled home centered in the frame, with a wing extending northwest and a chimney on the east side, and it sits in a forest clearing with no visible boundaries or driveway clearly reaching it.

2. **Tree overhang analysis:** Dense tree canopies press against the house on the east and southeast edges and along the southern wing, with an estimated 25-50% of the roof perimeter and edges under overhanging or adjacent foliage, while the central roof planes remain largely open.

3. **Fire risk assessment:** The overhanging canopy on the east and south sides creates a bridge from wildland vegetation to the roof and gutters, and the chimney and roof openings on the east side may be exposed to ember accumulation from nearby branches.

4. **Defensible space identification:** The trees form a nearly continuous canopy surrounding the home with little visible separation, and the dark foliag

**What this cell does:** Prints Claude's answer.

- `text_from_message(chat_response)` joins the text blocks of the response, so the thinking block is skipped.
- The output has one sentence for each of the five steps, then the final **Fire Risk Rating** with a short justification.
- Claude can only judge what is visible in the picture. A good answer, like the one saved here, says so when the image is too small or unclear to be sure.